# Flight Attribution

This notebook demonstrates how to fetch Google's L4 flight attributions from the contrails.org [Observations API](https://api.contrails.org/openapi#/Observations).

## Imports

In [34]:
%load_ext autoreload
%autoreload 2

import json
import os
import requests
import io
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

Set the flight identifier required for data retrieval.

To provide an API key, add it to a `.env` file in the same directory as this notebook with the contents:

```
CONTRAILS_API_KEY=<key here>
```

Contact api@contrails.org if you need an API key.

In [3]:
load_dotenv()
API_KEY = os.getenv("CONTRAILS_API_KEY")

if API_KEY:
    print("Key loaded successfully!")
else:
    print("Key not found. Check your .env file.")

Key loaded successfully!


In [12]:
# This cell demonstrates how to call the endpoint https://api.contrails.org/openapi#/Observations/geostationary_l4_attributions_v1_observations_google_geostationary_L4_attributions_get.
# This endpoint *requires* at least one 'flight identifier',
# which follows the following format:
# {carrier}~{flight_number}~{departure_date}~{departure_airport_iata}, e.g. AA~6441~2025-05-11~SAF.
# Each part of the flight identifier is required.
    
CARRIER = 'AF'                  # 2-character IATA airline designator (case-sensitive)
FLIGHT_NUMBER = '6113'          # Numeric portion of the flight number only (no carrier prefix)
DEPARTURE_DATE = '2025-02-12'   # Date, formatted as 'YYYY-MM-DD'.
DEPARTURE_AIRPORT_IATA = 'TLS'  # 3-character IATA code (case-sensitive)

flight_identifier = f'{CARRIER}~{FLIGHT_NUMBER}~{DEPARTURE_DATE}~{DEPARTURE_AIRPORT_IATA}'
print(f'Flight identifier: {flight_identifier}')

Flight identifier: AF~6113~2025-02-12~TLS


## Fetch flight attribution data

See [API documentation](https://api.contrails.org/openapi#/Observations/geostationary_l4_attributions_v1_observations_google_geostationary_L4_attributions_get).

In [13]:
URL = "https://api.contrails.org/v1/observations/google/geostationary/L4/attributions"
params = {
    'flight': [flight_identifier]
}
HEADERS = {
    'x-api-key': API_KEY
}

r = requests.get(URL, params=params, headers=HEADERS)
print(f"HTTP Response Code: {r.status_code} {r.reason}\n")

display(json.loads(r.content))

HTTP Response Code: 200 OK



[{'flight': 'AF~6113~2025-02-12~TLS',
  'attributions': [{'start_time': '2025-02-12T09:28:00Z',
    'end_time': '2025-02-12T09:41:00Z',
    'length_meters': 177895}],
  'observations': [{'time': '2025-02-12T09:40:07Z',
    'length_meters': 145326,
    'source': 'MTG_000_FULL_DISK'},
   {'time': '2025-02-12T09:50:07Z',
    'length_meters': 132192,
    'source': 'MTG_000_FULL_DISK'},
   {'time': '2025-02-12T10:10:07Z',
    'length_meters': 139347,
    'source': 'MTG_000_FULL_DISK'},
   {'time': '2025-02-12T10:20:07Z',
    'length_meters': 127282,
    'source': 'MTG_000_FULL_DISK'}]}]

## Understanding Flight Attributions Data

Each flight can have a number of attributions and observations.

### Attributions
Each attribution in `attributions` represents a distinct time interval of the flight in question to which contrail(s) were attributed.
- `start_time` represents the start time of the flight segment to which contrails are attributed.
- `end_time` represents the end time of the flight segment to which contrails are attributed.
- `length_meters` represents the length, in meters, of the subset(s) of the flight path to which contrails were attributed. This is calculated as the great circle distance between the flight waypoints.

### Observations 
Each observation in `observations` represents the satellite-based observation of the contrail in question.
- `time` is when the satellite image was taken in which the contrail was detected (typically the start time of the scan).
- `length_meters` is the observed end-to-end length of the linear contrail feature in meters, as detected in the satellite image specified by `time`. Note that this length can differ from the `length_meters` in `attributions` because:
  - The contrail may have evolved (e.g., spread, lengthened, etc.) in the atmosphere between formation and observation.
  - The attribution algorithm was only able to confidently attribute a portion of this contrail to this flight.
- `source` represents the specific satellite that observed the contrail in question. Available satellites are outlined in https://developers.google.com/contrails/reference/v2/attributions/overview#datasource, and each satellite's bounds are outlined in https://developers.google.com/contrails/reference/v2/attributions/overview#notes.

## Example use case with Flight Attributions Data

With the data from the attributions endpoint, it is possible to visualize which portions of a flight generated contrails using [Contrails.org's ADS-B telemetry endpoint](https://api.contrails.org/openapi#/ADS-B/adsb_telemetry_v1_adsb_telemetry_get), which is demonstrated below:

### Request Data

In [ ]:
ADSB_URL = "https://api.contrails.org/v1/adsb/telemetry"
HEADERS = {
    'accept': 'application/vnd.apache.parquet',
    'x-api-key': API_KEY
}
days = ["2025-02-12"]  # Can specify multiple days in case of a midnight flight
data = []
for day in days:
    for hour in range(24):
        params = {'date': f'{day}T{hour:02}'}
        r = requests.get(ADSB_URL, params=params, headers=HEADERS)
        print(f'Loaded data for {day}T{hour:02}')
        data.append(pd.read_parquet(io.BytesIO(r.content)))
flight_data = pd.concat(data)

### Filter data and plot the target flight

In [ ]:
target_flight = flight_data[(flight_data['flight_number'] == 'AF6113')]
# Times are determined from the 'attributions' field from attributions call for AF~6113~2025-02-12~TLS.
target_flight['is_attributed'] = (target_flight['timestamp'] >= '2025-02-12T09:28:00') & (target_flight['timestamp'] <= '2025-02-12T09:41:00')

def plot_flight_path(df):
  fig = px.scatter_geo(df,
                       lat='latitude', 
                       lon='longitude', 
                       color='is_attributed',
                       color_discrete_map={True: 'red', False: 'black'},
                       labels={'is_attributed': 'Attributed'},
                       title='Flight Path')
  fig.update_geos(showland=True, showocean=True, showcountries=True)
  fig.show()

plot_flight_path(target_flight)